# ARC-AGI-3 random-agent submission, explained

This is a **Kaggle competition submission notebook**, not the agent repository itself. It has two paths:

1. During Kaggle's hidden competition rerun, it writes the agent below, copies the supplied `ARC-AGI-3-Agents` repository into the writable working directory, and runs the agent against the local competition gateway.
2. During an ordinary interactive run, it creates a minimal placeholder `submission.parquet` instead. This lets Kaggle validate the notebook without attempting to contact the hidden evaluation service.

The `MyAgent` implementation is intentionally a baseline: it makes random action choices. Replace its decision logic with your own strategy to make a meaningful submission.

In [2]:
%%writefile working/my_agent.py
import random
import time
from typing import Any

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class MyAgent(Agent):
    """Random agent — picks a random action each step.
    
    Modify this class to implement your own agent strategy!
    """

    # The base Agent normally stops after 80 actions. This demo removes
    # that cap, so the run ends only after the game reaches WIN.
    MAX_ACTIONS = float('inf')

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        # Mix the current time and game ID to vary the random action stream.
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        # GAME_OVER is not terminal: choose_action will RESET and try again.
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            # Start a new attempt before making a move.
            action = GameAction.RESET
        else:
            # Once in progress, choose uniformly from every non-reset action.
            action = random.choice([a for a in GameAction if a is not GameAction.RESET])

        if action.is_simple():
            # Simple actions need no coordinates; reasoning is only metadata.
            action.reasoning = f"RNG told me to pick {action.value}"
        elif action.is_complex():
            # Complex actions need a position. This baseline guesses on a
            # 64×64 grid instead of reading the frame.
            action.set_data({
                "x": random.randint(0, 63),
                "y": random.randint(0, 63),
            })
            action.reasoning = {
                "desired_action": f"{action.value}",
                "my_reason": "RNG said so!",
            }
        return action

Overwriting working/my_agent.py


## What the agent methods mean

The framework repeatedly calls `choose_action(frames, latest_frame)`, sends the returned `GameAction` to the environment, and appends the resulting frame to `frames`. `latest_frame` contains the current game state, visual frame data, score, and available actions. `is_done(...)` controls when that loop ends.

For a real agent, `choose_action` is the main place to work: inspect `latest_frame.frame` and the action history in `frames`, then select an action and provide coordinates or other data when the chosen action is complex.

## Kaggle rerun setup

The competition evaluates this notebook by rerunning it in a controlled environment and exposing ARC games through `http://gateway:8001`. The next cell prepares the provided framework to use that private service, registers `MyAgent`, then invokes `main.py --agent myagent`.

The generated `.env` contains an evaluation-only gateway address and placeholder key. It is written inside Kaggle's temporary working directory and is not a credential to reuse outside that run.

In [3]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # This variable is set only by Kaggle's hidden scoring run. Wait until
    # its local ARC game server is ready to accept requests.
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Kaggle input datasets are read-only. Copy the supplied framework to
    # /kaggle/working so we can add our agent and configuration.
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # The first cell saved MyAgent to /kaggle/working/my_agent.py. Place it
    # in the template package so main.py can import it.
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Register the class under the CLI name "myagent". This replacement
    # initializer also avoids unused templates whose optional dependencies
    # may be absent from the Kaggle image.
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    # Point main.py to Kaggle's local gateway, not the public ARC service.
    # main.py loads .env after .env.example with override=True.
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # No --game filter means main.py runs the agent on every game exposed
    # by the gateway. MPLBACKEND=agg avoids needing a graphical backend.
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

## Interactive-run fallback

When you run the notebook outside Kaggle's hidden scoring pass, `KAGGLE_IS_COMPETITION_RERUN` is absent. The last cell therefore writes one syntactically valid, dummy row to the required `submission.parquet` path. Its values are not the agent's score and are not used to evaluate the strategy; actual evaluation occurs only in the rerun path above.

Writing Parquet requires `pyarrow` or `fastparquet`. The saved output below shows that neither was installed in the local kernel that last ran this cell; Kaggle's competition image must provide one for this fallback to finish.

In [6]:
# Outside the hidden evaluator, make a schema-valid placeholder file only.
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # These are the required columns, but fixed values here do not describe
    # an evaluated game.
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('working/submission.parquet', index=False)